In [1]:
import pandas as pd
import numpy as np

columns = [
    "engine_id", "cycle",
    "op_setting_1", "op_setting_2", "op_setting_3"
] + [f"sensor_{i}" for i in range(1, 22)]

test_df = pd.read_csv(
    "../dataset/test_FD001.txt",
    sep=r"\s+",
    header=None
)

test_df.columns = columns

true_rul = pd.read_csv(
    "../dataset/RUL_FD001.txt",
    header=None
).values.flatten()

print(test_df.shape)
print(true_rul.shape)

(13096, 26)
(100,)


In [3]:
feature_columns = [
    'cycle',
    'op_setting_1',
    'op_setting_2',
    'op_setting_3',
    'sensor_1',
    'sensor_2',
    'sensor_3',
    'sensor_4',
    'sensor_6',
    'sensor_7',
    'sensor_8',
    'sensor_9',
    'sensor_10',
    'sensor_11',
    'sensor_12',
    'sensor_13',
    'sensor_14',
    'sensor_15',
    'sensor_17',
    'sensor_19',
    'sensor_20',
    'sensor_21'
]

print(len(feature_columns))

22


In [8]:
X_test = []
test_engine_ids = []

for engine_id, engine in test_df.groupby("engine_id"):

    engine = engine.sort_values("cycle")
    values = engine[feature_columns].values

    if len(values) >= 100:
        sequence = values[-100:]

    else:
        padding = np.repeat(
            values[0:1],
            100 - len(values),
            axis=0
        )
        sequence = np.vstack([padding, values])

    X_test.append(sequence)
    test_engine_ids.append(engine_id)

X_test = np.array(X_test)

print(X_test.shape)

(100, 100, 22)


In [7]:
print("Test rows:", test_df.shape)
print("Test engines:", test_df["engine_id"].nunique())
print("Official RUL values:", len(true_rul))

print(
    test_df.groupby("engine_id")["cycle"]
    .max()
    .describe()
)

Test rows: (13096, 26)
Test engines: 100
Official RUL values: 100
count    100.000000
mean     130.960000
std       53.593479
min       31.000000
25%       88.750000
50%      133.500000
75%      164.250000
max      303.000000
Name: cycle, dtype: float64


In [10]:
from sklearn.preprocessing import StandardScaler

scaler100 = StandardScaler()

n_train, seq_len, n_features = X_train_100.shape

X_train_100_2d = X_train_100.reshape(-1, n_features)

scaler100.fit(X_train_100_2d)

print("Scaler fitted on training data.")

NameError: name 'X_train_100' is not defined

In [11]:
print("lstm100 exists:", "lstm100" in globals())
print("X_train_100 exists:", "X_train_100" in globals())
print("scaler100 exists:", "scaler100" in globals())
print("X_test exists:", "X_test" in globals())

lstm100 exists: False
X_train_100 exists: False
scaler100 exists: True
X_test exists: True


In [12]:
import os

print(os.listdir())

['01_dataset_exploration.ipynb', '02_data_preprocessing.ipynb', '03_ml_models.ipynb', '04_dl_models.ipynb']


In [13]:
X_train_100, y_train_100 = make_sequences(
    train_seq, feature_columns, 100
)

X_val_100, y_val_100 = make_sequences(
    val_seq, feature_columns, 100
)

print(X_train_100.shape)
print(X_val_100.shape)

NameError: name 'make_sequences' is not defined

In [14]:
from tensorflow.keras.models import load_model
import joblib

lstm100 = load_model("../lstm100.keras")
scaler100 = joblib.load("../scaler100.pkl")

In [15]:
X_test_2d = X_test.reshape(-1, 22)

X_test_scaled = scaler100.transform(X_test_2d).reshape(
    100, 100, 22
)

In [16]:
test_pred = lstm100.predict(X_test_scaled).ravel()

print(test_pred[:10])

4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 176ms/step
[202.36687  111.40765   71.1776    68.8393    90.08559  132.28236
 103.608055 110.36994  114.077286 130.29007 ]


In [17]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("Test MAE:", mean_absolute_error(true_rul, test_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(true_rul, test_pred)))
print("Test R²:", r2_score(true_rul, test_pred))

Test MAE: 17.60298728942871
Test RMSE: 27.726890495383156
Test R²: 0.5548126697540283


In [18]:
long_ids = [
    engine_id
    for engine_id, engine in test_df.groupby("engine_id")
    if len(engine) >= 100
]

print("Engines with >=100 cycles:", len(long_ids))

Engines with >=100 cycles: 70


In [20]:
mask = np.isin(test_engine_ids, long_ids)

pred_70 = test_pred[mask]

engine_ids_70 = np.array(test_engine_ids)[mask]

# Engine ID 1 → index 0, Engine ID 100 → index 99
rul_70 = true_rul[engine_ids_70.astype(int) - 1]

print("Engines:", len(rul_70))
print("MAE:", mean_absolute_error(rul_70, pred_70))
print("RMSE:", np.sqrt(mean_squared_error(rul_70, pred_70)))
print("R²:", r2_score(rul_70, pred_70))

Engines: 70
MAE: 10.182453155517578
RMSE: 15.550105172442514
R²: 0.8375542163848877


In [21]:
test_df = pd.read_csv(
    "../dataset/test_FD001.txt",
    sep=r"\s+",
    header=None
)

test_df.columns = columns

true_rul = pd.read_csv(
    "../dataset/RUL_FD001.txt",
    header=None
).values.flatten()

In [22]:
X_test = []

for _, engine in test_df.groupby("engine_id"):
    engine = engine.sort_values("cycle")
    values = engine[feature_columns].values

    if len(values) >= 100:
        sequence = values[-100:]
    else:
        padding = np.zeros((100 - len(values), 22))
        sequence = np.vstack([padding, values])

    X_test.append(sequence)

X_test = np.array(X_test)

print(X_test.shape)

(100, 100, 22)


In [24]:
import joblib

scaler = joblib.load("../scaler_final.pkl")

print("Scaler loaded")

X_test_scaled = scaler.transform(
    X_test.reshape(-1, 22)
).reshape(100, 100, 22)

X_test_scaled[X_test == 0] = -999

Scaler loaded


c:\Users\Lenovo\Desktop\aeroguardian\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [26]:
from tensorflow.keras.models import load_model
import joblib

lstm_final = load_model("../lstm_final.keras")
test_pred = lstm_final.predict(X_test_scaled).ravel()

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 193ms/step


In [27]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("Test MAE:", mean_absolute_error(true_rul, test_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(true_rul, test_pred)))
print("Test R²:", r2_score(true_rul, test_pred))

Test MAE: 20.361957550048828
Test RMSE: 27.62717508582186
Test R²: 0.5580090284347534
